# Photometric robustness benchmark — reference metric + mixed loader

Corrected preprocessing setup:

- SegFormer / Mask2Former:
  - use `pair_raw`, i.e. RGB tensors in `[0,1]`;
  - use the HuggingFace processor path inside their adapters.

- DeepLabV3+ / PIDNet / DDRNet:
  - use `pair_repo`, i.e. RGB tensors normalized with ImageNet mean/std in the dataloader;
  - use `input_norm: none` inside their adapters to avoid double normalization.

Metric:
- clean pseudo-labels are generated by the reference model;
- adverse predictions are generated by each evaluated model;
- retention/agreement are computed against the reference clean pseudo-label.


In [ ]:
from pathlib import Path

CONFIG_PATH = Path("./benchmark_config_reference_metric_mixed_loader.yaml")
assert CONFIG_PATH.exists(), f"Config file not found: {CONFIG_PATH}"
print("Using config:", CONFIG_PATH.resolve())


In [ ]:
import os
import sys
import json
import copy
import random
from collections import defaultdict
from typing import Any, Dict, List

import yaml
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

for p in [
    "/home/ace/Downloads",
    "/home/ace/Downloads/PIDNet",
    "/home/ace/Downloads/pytorch-semseg",
    "/home/ace/Downloads/DeepLabV3PlusPytorch",
    "/home/ace/Downloads/DDRNet",
    "/home/ace/Downloads/DDRNet/segmentation",
]:
    if p not in sys.path:
        sys.path.append(p)

from dataloader_benchmark_mixed import PairedImageDataset

from benchmark_models_mixed_loader import (
    BaseAdapter,
    build_adapter_from_spec,
    resolve_model_spec_from_key,
)


In [ ]:
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

cfg


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.get("seed", 0))

DATA_DIR = cfg["data_dir"]
SAVE_DIR = Path(cfg["save_dir"])
DEVICE = cfg.get("device", "cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = int(cfg.get("batch_size", 2))
NUM_WORKERS = int(cfg.get("num_workers", 4))
CONF_THRESHOLD = float(cfg.get("conf_threshold", 0.6))
MASS_COVERAGE = float(cfg.get("mass_coverage", 0.99))
REFERENCE_MODEL_KEY = cfg.get("reference_model_key", "segformer_b5")
REFERENCE_INPUT_SOURCE = cfg.get("reference_input_source", "raw")
REFERENCE_INPUT_NORM = cfg.get("reference_input_norm", "processor")
DATALOADER_MODE = cfg.get("dataloader_mode", "random")
PLOT_SCENARIOS = list(cfg.get("plot_scenarios", ["Day-RAIN", "Sunset-FOGGY", "Night-RAIN"]))
PLOT_DROP_CLASSNAME = cfg.get("plot_drop_classname", "train")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR.resolve())
print("DEVICE:", DEVICE)
print("REFERENCE_MODEL_KEY:", REFERENCE_MODEL_KEY)
print("REFERENCE_INPUT_SOURCE:", REFERENCE_INPUT_SOURCE)


In [ ]:
def normalize_model_config_with_source(models_cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Resolve model specs while preserving input_source metadata from YAML."""
    out = []
    for item in models_cfg.get("enabled", []):
        if isinstance(item, str):
            spec = resolve_model_spec_from_key(item)
            spec["input_source"] = "raw" if spec["type"] in {"segformer", "mask2former"} else "repo"
            out.append(spec)
        elif isinstance(item, dict):
            if "key" in item:
                spec = resolve_model_spec_from_key(item["key"])
                spec.update({k: v for k, v in item.items() if k != "key"})
                if "input_source" not in spec:
                    spec["input_source"] = "raw" if spec["type"] in {"segformer", "mask2former"} else "repo"
                out.append(spec)
            else:
                spec = dict(item)
                if "input_source" not in spec:
                    spec["input_source"] = "raw" if spec["type"] in {"segformer", "mask2former"} else "repo"
                out.append(spec)
        else:
            raise TypeError("Each entry in models.enabled must be a string or dict.")
    return out

def build_adapters_with_specs(models_cfg, device):
    specs = normalize_model_config_with_source(models_cfg)
    adapters = []
    for spec in specs:
        adapters.append(build_adapter_from_spec(spec, device))
    return list(zip(adapters, specs))

def select_pair_tensor(batch, source: str):
    source = str(source).lower()
    if source == "raw":
        return batch["pair_raw"]
    if source in {"repo", "repo_imagenet", "imagenet"}:
        return batch["pair_repo"]
    raise ValueError(f"Unknown input_source: {source}")


In [ ]:
def pixel_agreement_masked(pred_ref: torch.Tensor, pred_model: torch.Tensor, mask_ref: torch.Tensor) -> torch.Tensor:
    same = (pred_ref == pred_model) & mask_ref
    denom = mask_ref.flatten(1).sum(dim=1).clamp_min(1)
    num = same.flatten(1).sum(dim=1)
    return num.double() / denom.double()

def batch_counts_by_condition(pred_ref, pred_model_adv, ref_mask, conditions, num_classes: int):
    idx_by_cond = defaultdict(list)
    for i, c in enumerate(conditions):
        idx_by_cond[c].append(i)

    out = {}
    for c, idxs in idx_by_cond.items():
        idxs_t = torch.tensor(idxs, device=pred_ref.device, dtype=torch.long)

        a = pred_ref.index_select(0, idxs_t)
        b = pred_model_adv.index_select(0, idxs_t)
        m = ref_mask.index_select(0, idxs_t)

        a_f = a[m]
        b_f = b[m]

        valid = (a_f >= 0) & (a_f < num_classes) & (b_f >= 0) & (b_f < num_classes)
        a_f = a_f[valid]
        b_f = b_f[valid]

        mass = torch.bincount(a_f, minlength=num_classes).to(torch.float64)
        same = torch.bincount(a_f[a_f == b_f], minlength=num_classes).to(torch.float64)
        out[c] = (same.detach().cpu(), mass.detach().cpu())

    return out

def select_classes_by_mass_coverage(df_s: pd.DataFrame, coverage: float, drop_class: str = None):
    df = df_s.copy()
    if drop_class is not None:
        df = df[df["class_name"] != drop_class]

    mass_by_class = df.groupby("class_name")["mass_clean_masked"].sum().sort_values(ascending=False)
    total = float(mass_by_class.sum())

    if total < 1e-9:
        return []

    cum = mass_by_class.cumsum() / total
    keep = mass_by_class.index[cum <= coverage].tolist()

    if len(keep) == 0:
        keep = [mass_by_class.index[0]]
    elif cum.iloc[len(keep) - 1] < coverage and len(keep) < len(mass_by_class):
        keep.append(mass_by_class.index[len(keep)])

    return keep


In [ ]:
@torch.no_grad()
def run_benchmark():
    device = torch.device(DEVICE)

    adapter_specs = build_adapters_with_specs(cfg.get("models"), device)

    if not adapter_specs:
        raise RuntimeError("No models enabled in the YAML configuration.")

    ref_spec = resolve_model_spec_from_key(REFERENCE_MODEL_KEY)
    ref_spec["input_source"] = REFERENCE_INPUT_SOURCE
    # For HF reference model, keep processor path.
    ref_adapter = build_adapter_from_spec(ref_spec, device)

    print(f"Reference model: {ref_adapter.name} | input_source={REFERENCE_INPUT_SOURCE}")

    id2label = ref_adapter.id2label() or {i: str(i) for i in range(ref_adapter.num_classes())}
    K = len(id2label)
    class_names = [id2label[i] for i in range(K)]

    print("Enabled models:")
    for adp, spec in adapter_specs:
        print(f" - {adp.name} | input_source={spec.get('input_source')} | input_norm={getattr(adp, 'input_norm', 'processor')}")

    ds = PairedImageDataset(DATA_DIR, mode=DATALOADER_MODE)

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    rows_img = []
    cond_sum = {a.name: defaultdict(float) for a, _ in adapter_specs}
    cond_n = {a.name: defaultdict(int) for a, _ in adapter_specs}
    cond_same = {a.name: defaultdict(lambda: torch.zeros(K, dtype=torch.float64)) for a, _ in adapter_specs}
    cond_mass = {a.name: defaultdict(lambda: torch.zeros(K, dtype=torch.float64)) for a, _ in adapter_specs}
    global_same = {a.name: torch.zeros(K, dtype=torch.float64) for a, _ in adapter_specs}
    global_mass = {a.name: torch.zeros(K, dtype=torch.float64) for a, _ in adapter_specs}

    for batch in tqdm(loader, desc="Evaluating"):
        ref_pair = select_pair_tensor(batch, REFERENCE_INPUT_SOURCE)
        clean_ref = ref_pair[:, 0].to(device, non_blocking=True)

        loc_ids = batch["loc_id"]
        conds = [c if c is not None else "UNKNOWN" for c in batch["sec_condition"]]

        pred_ref_clean, conf_ref_clean = ref_adapter.predict(clean_ref)
        ref_mask = conf_ref_clean >= CONF_THRESHOLD

        for adp, spec in adapter_specs:
            model_pair = select_pair_tensor(batch, spec.get("input_source", "raw"))
            adv_model = model_pair[:, 1].to(device, non_blocking=True)

            pred_adv, _ = adp.predict(adv_model)

            agree = pixel_agreement_masked(pred_ref_clean, pred_adv, ref_mask)

            counts = batch_counts_by_condition(pred_ref_clean, pred_adv, ref_mask, conds, num_classes=K)
            for c, (same_c, mass_c) in counts.items():
                cond_same[adp.name][c] += same_c
                cond_mass[adp.name][c] += mass_c
                global_same[adp.name] += same_c
                global_mass[adp.name] += mass_c

            for i in range(adv_model.shape[0]):
                c = conds[i]
                v = float(agree[i].item())
                cond_sum[adp.name][c] += v
                cond_n[adp.name][c] += 1
                coverage = float(ref_mask[i].float().mean().item())

                rows_img.append({
                    "model": adp.name,
                    "reference_model": ref_adapter.name,
                    "loc_id": loc_ids[i],
                    "sec_condition": c,
                    "pixel_agreement_masked": v,
                    "metric_type": "reference_clean_vs_model_adverse",
                    "reference_input_source": REFERENCE_INPUT_SOURCE,
                    "model_input_source": spec.get("input_source"),
                    "adapter_input_norm": getattr(adp, "input_norm", "processor"),
                    "conf_threshold": CONF_THRESHOLD,
                    "coverage_masked": coverage,
                    "coverage_ref_masked": coverage,
                    "clean_path": batch["clean_path"][i],
                    "sec_path": batch["sec_path"][i],
                })

    df_img = pd.DataFrame(rows_img)
    df_img.to_csv(SAVE_DIR / "results_pixel_agreement_per_image.csv", index=False)

    rows_cond = []
    for m in cond_sum:
        for c in cond_sum[m]:
            n = cond_n[m][c]
            rows_cond.append({
                "model": m,
                "reference_model": ref_adapter.name,
                "sec_condition": c,
                "pixel_agreement_masked_mean": cond_sum[m][c] / max(n, 1),
                "metric_type": "reference_clean_vs_model_adverse",
                "n_pairs": int(n),
                "conf_threshold": CONF_THRESHOLD,
            })

    df_cond = pd.DataFrame(rows_cond)
    df_cond.to_csv(SAVE_DIR / "results_pixel_agreement_by_condition.csv", index=False)

    rows_cls_cond = []
    for m in cond_same:
        for c in cond_same[m]:
            mass = cond_mass[m][c].clamp_min(1.0)
            ret = cond_same[m][c] / mass
            for k in range(K):
                rows_cls_cond.append({
                    "model": m,
                    "reference_model": ref_adapter.name,
                    "sec_condition": c,
                    "class_id": k,
                    "class_name": class_names[k],
                    "retention_masked": float(ret[k].item()),
                    "mass_clean_masked": float(cond_mass[m][c][k].item()),
                    "mass_reference_masked": float(cond_mass[m][c][k].item()),
                    "metric_type": "reference_clean_vs_model_adverse",
                    "conf_threshold": CONF_THRESHOLD,
                })

    df_cls_cond = pd.DataFrame(rows_cls_cond)
    df_cls_cond.to_csv(SAVE_DIR / "results_class_retention_by_condition.csv", index=False)

    rows_cls_global = []
    for m in global_same:
        mass = global_mass[m].clamp_min(1.0)
        ret = global_same[m] / mass
        for k in range(K):
            rows_cls_global.append({
                "model": m,
                "reference_model": ref_adapter.name,
                "class_id": k,
                "class_name": class_names[k],
                "retention_masked_global": float(ret[k].item()),
                "mass_clean_masked_global": float(global_mass[m][k].item()),
                "mass_reference_masked_global": float(global_mass[m][k].item()),
                "metric_type": "reference_clean_vs_model_adverse",
                "conf_threshold": CONF_THRESHOLD,
            })

    df_cls_global = pd.DataFrame(rows_cls_global)
    df_cls_global.to_csv(SAVE_DIR / "results_class_retention_global.csv", index=False)

    return adapter_specs, df_img, df_cond, df_cls_cond, df_cls_global


In [ ]:
adapter_specs, df_img, df_cond, df_cls_cond, df_cls_global = run_benchmark()
print("Saved benchmark outputs to:", SAVE_DIR.resolve())
df_cond.head()


In [ ]:
df_cond.pivot_table(
    index="model",
    columns="sec_condition",
    values="pixel_agreement_masked_mean"
).round(4)


In [ ]:
plot_df = df_cond[df_cond["sec_condition"].isin(PLOT_SCENARIOS)].copy()

if not plot_df.empty:
    pivot = plot_df.pivot(index="model", columns="sec_condition", values="pixel_agreement_masked_mean")
    pivot = pivot.reindex(columns=[c for c in PLOT_SCENARIOS if c in pivot.columns])

    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_ylabel("Reference-masked agreement")
    ax.set_xlabel("Model")
    ax.set_title("Agreement between reference clean pseudo-label and model adverse prediction")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / "plot_reference_agreement_by_condition.pdf", bbox_inches="tight")
    plt.savefig(SAVE_DIR / "plot_reference_agreement_by_condition.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No selected scenarios available in df_cond.")


In [ ]:
for scenario in PLOT_SCENARIOS:
    df_s = df_cls_cond[df_cls_cond["sec_condition"] == scenario].copy()

    if df_s.empty:
        print(f"Skipping {scenario}: no data")
        continue

    keep_classes = select_classes_by_mass_coverage(
        df_s,
        coverage=MASS_COVERAGE,
        drop_class=PLOT_DROP_CLASSNAME,
    )

    df_p = df_s[df_s["class_name"].isin(keep_classes)].copy()

    if df_p.empty:
        print(f"Skipping {scenario}: no selected classes")
        continue

    pivot = df_p.pivot_table(
        index="class_name",
        columns="model",
        values="retention_masked",
        aggfunc="mean"
    )

    mass_order = (
        df_p.groupby("class_name")["mass_clean_masked"]
        .sum()
        .sort_values(ascending=False)
        .index
    )

    pivot = pivot.reindex(mass_order)

    ax = pivot.plot(kind="bar", figsize=(14, 5), width=0.85)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Reference-based retention")
    ax.set_xlabel("Class")
    ax.set_title(f"Class-wise reference-based retention — {scenario}")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fname = f"plot_bar_class_retention_reference__{scenario}.pdf".replace("/", "_")
    plt.savefig(SAVE_DIR / fname, bbox_inches="tight")
    plt.savefig(SAVE_DIR / fname.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
with open(SAVE_DIR / "run_config_resolved.json", "w") as f:
    json.dump({
        "data_dir": DATA_DIR,
        "save_dir": str(SAVE_DIR),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "conf_threshold": CONF_THRESHOLD,
        "mass_coverage": MASS_COVERAGE,
        "plot_scenarios": PLOT_SCENARIOS,
        "drop_class_from_plot": PLOT_DROP_CLASSNAME,
        "models": [a.name for a, _ in adapter_specs],
        "reference_model_key": REFERENCE_MODEL_KEY,
        "reference_input_source": REFERENCE_INPUT_SOURCE,
        "dataloader_mode": DATALOADER_MODE,
        "metric_type": "reference_clean_vs_model_adverse",
        "preprocessing_note": cfg.get("preprocessing_note"),
    }, f, indent=2)

print("Saved:", SAVE_DIR / "run_config_resolved.json")
